# Pluto: test the latest text checkpoint and the voice model

Run these cells from top to bottom in Google Colab. This notebook loads **two separate checkpoints**: the M3 text model and the existing M2 speech model. M3 can write text; M2 turns text into audio. The optional final cell passes M3's text to M2 for speech. This is inference only: it does not train, edit, or merge model weights.

The model is still experimental, so generated text and speech may be weak or malformed. The M2 speech decoder uses the 12-frame pacing guard from the project's decoding experiment.


## 1. Start a free GPU runtime

In Colab choose **Runtime → Change runtime type → T4 GPU → Save**. Then run this check.


In [ ]:
import shutil
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("A GPU is not attached. In Colab, choose Runtime → Change runtime type → T4 GPU, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)


## 2. Add your private Hub token

In Colab, open the **key icon (Secrets)** in the left sidebar, add a secret named `HF_TOKEN`, paste your Hugging Face read token there, and enable notebook access. Do not put the token in a code cell.


## 3. Clone Pluto and install inference dependencies

This installs the Pluto source and the speech codec library into Colab's current Python. It leaves Colab's CUDA-enabled PyTorch installation in place.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add HF_TOKEN in Colab Secrets and enable notebook access, then rerun this cell.")
os.environ["HF_TOKEN"] = HF_TOKEN

REPO_URL = "https://github.com/thedeepakreddy/Pluto-LLM.git"
WORK = Path("/content/Pluto-LLM")
if not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(WORK)], check=True)
else:
    subprocess.run(["git", "-C", str(WORK), "fetch", "-q", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(WORK), "reset", "--hard", "origin/main"], check=True)
os.chdir(WORK)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "transformers", "PyYAML", "regex", "datasets"], check=True)
print("Pluto installed from", WORK)
print("PyTorch", torch.__version__, "GPU:", torch.cuda.get_device_name(0))


## 4. Choose the checkpoints

The M3 run is found automatically by its newest run name that has uploaded a `latest.json`. Set `TEXT_RUN_ID_OVERRIDE` if you want another M3 run. The M2 voice run is set to the existing aligned speech checkpoint; change `VOICE_RUN_ID` only if you want a different M2 run.


In [ ]:
import re
from huggingface_hub import HfApi

api = HfApi()
HUB_REPO = api.whoami()["name"] + "/pluto-runs"
REPO_FILES = api.list_repo_files(repo_id=HUB_REPO, repo_type="model")

TEXT_RUN_ID_OVERRIDE = ""  # Optional: paste an M3 run ID here to override automatic selection.
VOICE_RUN_ID = "m2-aligned-20261102-213616-72fb023"

m3_runs = sorted({
    path.split("/", 1)[0]
    for path in REPO_FILES
    if path.startswith("m3-pilot-") and path.endswith("/checkpoints/latest.json")
}, reverse=True)
if TEXT_RUN_ID_OVERRIDE:
    TEXT_RUN_ID = TEXT_RUN_ID_OVERRIDE.strip()
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", TEXT_RUN_ID):
        raise ValueError("TEXT_RUN_ID_OVERRIDE must be a plain run ID, without slashes.")
    if f"{TEXT_RUN_ID}/checkpoints/latest.json" not in REPO_FILES:
        raise FileNotFoundError(f"No uploaded latest checkpoint found for {TEXT_RUN_ID} in {HUB_REPO}.")
else:
    if not m3_runs:
        raise FileNotFoundError("No uploaded m3-pilot checkpoint was found in the private Hub repo yet.")
    TEXT_RUN_ID = m3_runs[0]

if f"{VOICE_RUN_ID}/checkpoints/latest.json" not in REPO_FILES:
    raise FileNotFoundError(f"The selected M2 voice run {VOICE_RUN_ID} is not in {HUB_REPO}. Edit VOICE_RUN_ID to a run that has an uploaded latest.json.")

print("Private Hub repo:", HUB_REPO)
print("M3 text run:", TEXT_RUN_ID)
print("M2 voice run:", VOICE_RUN_ID)


## 5. Download only each run's latest checkpoint

A run can have several large training checkpoints. This downloads only `latest.json`, the current checkpoint named by it, and the run config. It does not download old checkpoints or upload anything.


In [ ]:
import json
from huggingface_hub import snapshot_download


def download_latest_checkpoint(run_id):
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", run_id):
        raise ValueError("Run ID must be a plain name, without slashes.")
    pointer = f"{run_id}/checkpoints/latest.json"
    config_file = f"{run_id}/config.yaml"
    snapshot_download(
        repo_id=HUB_REPO,
        repo_type="model",
        allow_patterns=[pointer, config_file],
        local_dir=str(WORK),
    )
    run_dir = WORK / run_id
    latest = json.loads((run_dir / "checkpoints" / "latest.json").read_text())
    checkpoint_name = latest["path"]
    if Path(checkpoint_name).name != checkpoint_name or not checkpoint_name.endswith(".pt"):
        raise ValueError(f"Unexpected checkpoint path in latest.json: {checkpoint_name!r}")
    checkpoint_repo_path = f"{run_id}/checkpoints/{checkpoint_name}"
    snapshot_download(
        repo_id=HUB_REPO,
        repo_type="model",
        allow_patterns=[checkpoint_repo_path],
        local_dir=str(WORK),
    )
    checkpoint_path = run_dir / "checkpoints" / checkpoint_name
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Hub did not provide {checkpoint_repo_path}.")
    return run_dir, checkpoint_path, int(latest["step"])


TEXT_RUN_DIR, TEXT_CHECKPOINT, TEXT_CHECKPOINT_STEP = download_latest_checkpoint(TEXT_RUN_ID)
VOICE_RUN_DIR, VOICE_CHECKPOINT, VOICE_CHECKPOINT_STEP = download_latest_checkpoint(VOICE_RUN_ID)
print(f"M3 checkpoint step: {TEXT_CHECKPOINT_STEP}")
print(f"M2 checkpoint step: {VOICE_CHECKPOINT_STEP}")


## 6. Prepare the M2 tokenizer if it is missing

The M2 tokenizer is separate from M3's tokenizer. If it is not already present in this Colab runtime, this rebuilds it from the project's public TinyStories recipe. That CPU setup may take around 15–40 minutes. It prepares text data and the tokenizer only; it does **not** build token shards or train a model.


In [ ]:
from llm.config import load_config
from llm.data.prepare import data_dir, prepare_data
from llm.manifest import ProvenanceError, verify_files
from llm.tokenizer.bpe import tokenizer_dir, train_tokenizer

voice_run_cfg = load_config(VOICE_RUN_DIR / "config.yaml")
m2_tokenizer_dir = tokenizer_dir(voice_run_cfg)


def artifact_is_valid(path):
    try:
        verify_files(path)
        return True
    except (ProvenanceError, FileNotFoundError, ValueError):
        return False


if artifact_is_valid(m2_tokenizer_dir):
    print("M2 tokenizer already exists:", m2_tokenizer_dir)
else:
    warm_cfg = load_config("configs/m2-warmstart.yaml")
    if voice_run_cfg.tokenizer != warm_cfg.tokenizer:
        raise RuntimeError("The selected M2 run uses a different tokenizer recipe. Do not rebuild it from m2-warmstart.yaml.")
    warm_data_dir = data_dir(warm_cfg)
    if not artifact_is_valid(warm_data_dir):
        print("Preparing public TinyStories data for the M2 tokenizer. This can take a while...")
        prepare_data(warm_cfg)
    print("Training the M2 tokenizer from the prepared TinyStories text...")
    train_tokenizer(warm_cfg)
    if not artifact_is_valid(m2_tokenizer_dir):
        raise RuntimeError("M2 tokenizer build finished but its manifest/checksum validation failed.")
    print("M2 tokenizer ready:", m2_tokenizer_dir)


## 7. Edit the voice prompt

Change `VOICE_PROMPT` below to the sentence you want the M2 speech model to read aloud.


In [ ]:
VOICE_PROMPT = "Hello! I am testing Pluto's voice model."  # EDIT THIS SENTENCE
VOICE_TEMPERATURE = 0.8
VOICE_TOP_K = 50
VOICE_SEED = 1234
MAX_PAD_RUN = 12  # decoder pacing guard: about one second at 12.5 frames/second


## 8. Generate and play speech with M2

This uses the M2 speech checkpoint and the off-the-shelf Kyutai Mimi codec to decode audio. It does not use the M3 text checkpoint.


In [ ]:
from IPython.display import Audio, display
from llm.speech.generate import SpeechSampling, speak
from llm.speech.interfaces import make_codec
from llm.train.checkpoint import load_speech_model_for_inference

voice_model, voice_tok, voice_cfg = load_speech_model_for_inference(VOICE_CHECKPOINT)
voice_model = voice_model.to("cuda").eval()
codec = make_codec(voice_cfg.speech)
voice_codes = speak(
    voice_model,
    VOICE_PROMPT,
    voice_tok,
    voice_cfg.audio,
    SpeechSampling(
        temperature=VOICE_TEMPERATURE,
        top_k=VOICE_TOP_K,
        seed=VOICE_SEED,
        max_pad_run=MAX_PAD_RUN,
    ),
)
voice_audio = codec.decode(voice_codes)
print(f"M2 generated {len(voice_codes)} audio frames ({len(voice_audio) / codec.sample_rate:.1f} seconds).")
display(Audio(voice_audio, rate=codec.sample_rate))


## 9. Get the M3 tokenizer

The M3 checkpoint does not contain its tokenizer. If its tokenizer is already present in this runtime, the cell validates it. Otherwise it looks for the private Hub artifact at `tokenizers/m3-bpe-v1`.

If the artifact has not been uploaded yet, this cell stops with instructions. The upload is a separate CPU-only Modal step; because your Modal balance is low, review the remaining credit before running it.


In [ ]:
from huggingface_hub import snapshot_download
from llm.config import load_config
from llm.tokenizer.bpe import tokenizer_dir

m3_cfg = load_config("configs/m3-pilot.yaml")
m3_tokenizer_dir = tokenizer_dir(m3_cfg)
m3_tokenizer_remote = m3_tokenizer_dir.as_posix()

if not artifact_is_valid(m3_tokenizer_dir):
    remote_files = [
        path for path in REPO_FILES
        if path.startswith(m3_tokenizer_remote + "/")
    ]
    if not remote_files:
        raise RuntimeError(
            "The M3 tokenizer is not published to the private Hub repo yet. After reviewing the Modal credit cost, run these on your Mac once the training session is stopped:\n\n"
            "PLUTO_GPU=H100 modal deploy modal_app/m3_train.py\n"
            "modal run modal_app/m3_train.py --steps publish-tokenizer --spawn"
        )
    snapshot_download(
        repo_id=HUB_REPO,
        repo_type="model",
        allow_patterns=[m3_tokenizer_remote + "/*"],
        local_dir=str(WORK),
    )
if not artifact_is_valid(m3_tokenizer_dir):
    raise RuntimeError("The downloaded M3 tokenizer failed its manifest/checksum validation.")
print("M3 tokenizer ready:", m3_tokenizer_dir)


## 10. Edit the text prompt

Change `TEXT_PROMPT` to the prompt you want the M3 text model to continue. These settings control how many tokens it writes and how random its choices are.


In [ ]:
TEXT_PROMPT = "Explain why the sky looks blue in simple terms:"  # EDIT THIS PROMPT
MAX_NEW_TOKENS = 120
TEXT_TEMPERATURE = 0.8
TEXT_TOP_K = 50
TEXT_SEED = 1234


## 11. Generate text with M3

This uses the selected M3 text checkpoint. The model runs on Colab's GPU.


In [ ]:
from llm.generate import generate
from llm.train.checkpoint import load_model_for_inference

text_model, text_tok, text_payload = load_model_for_inference(TEXT_CHECKPOINT)
text_model = text_model.to("cuda").eval()
del text_payload  # The training checkpoint also contains optimizer state; inference does not need it.
new_ids = generate(
    text_model,
    text_tok.encode(TEXT_PROMPT),
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEXT_TEMPERATURE,
    top_k=TEXT_TOP_K,
    stop_ids=[text_tok.eot_id],
    seed=TEXT_SEED,
)
TEXT_OUTPUT = text_tok.decode(new_ids)
print("M3 text output:\n")
print(TEXT_OUTPUT)


## 12. Optional: have M2 speak the text M3 generated

This is a two-model test: M3 writes the text, then M2 reads that text aloud. The voice model remains the M2 checkpoint.


In [ ]:
CHAINED_TEXT = TEXT_OUTPUT.strip()
if not CHAINED_TEXT:
    raise RuntimeError("M3 produced no text to send to the voice model. Rerun text generation or edit CHAINED_TEXT.")

chained_codes = speak(
    voice_model,
    CHAINED_TEXT,
    voice_tok,
    voice_cfg.audio,
    SpeechSampling(
        temperature=VOICE_TEMPERATURE,
        top_k=VOICE_TOP_K,
        seed=VOICE_SEED,
        max_pad_run=MAX_PAD_RUN,
    ),
)
chained_audio = codec.decode(chained_codes)
print("M2 speech of the M3-generated text:")
display(Audio(chained_audio, rate=codec.sample_rate))


## Troubleshooting

- **No GPU:** reconnect with a T4 runtime, then rerun from the first cell.
- **Private Hub access denied:** confirm `HF_TOKEN` is a read token with access to your private `pluto-runs` repo.
- **No M3 run found:** the H100 run must have uploaded a checkpoint. Confirm a run with `checkpoints/latest.json` exists in the private repo.
- **M3 tokenizer not published:** the text test needs the M3 tokenizer artifact. The upload commands are printed in the tokenizer cell; they use Modal CPU credits.
- **M2 setup is taking time:** if the M2 tokenizer is absent, the notebook prepares public TinyStories text and recreates that tokenizer. It does not train the M2 model.
- **Out of GPU memory:** restart the Colab runtime, choose a T4 GPU, and use a shorter prompt with smaller generation limits.
- **Poor or odd output:** these are small experimental models; this notebook reports raw generations and does not claim a quality score.
